### Phát biểu bài toán:

-   Phân tích mức độ ảnh hưởng của phần cứng và thương hiệu đến giá của điện thoại di động

### Nguồn Dữ liệu:

-   Dữ liệu được thu thập từ trang web:
    -   https://mobilecity.vn/dien-thoai
    -   https://cellphones.com.vn/mobile.html

Phân tích dữ liệu nhằm khảo sát tính khả thi cho việc xây dựng mô hình dự đoán biến mục tiêu (target variable) "Giá - Price" từ các biến/đặc trưng phần cứng và thương hiệu Xi (i=1..N).
Vì biến đăng trưng Y là biến số liên tục (continuous variable) -> việc mô hình hóa là bài toán hồi quy (regression).


# Bước 1: Khai thác dữ liệu (Data Exploration)


### Import các thư viện cần thiết


In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.webdriver import WebDriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time
from typing import Any
from concurrent.futures import ThreadPoolExecutor, wait, as_completed
from threading import Lock
import csv
import os
import json
import random
from tqdm import tqdm

## Đối với trang web cellphones.com.vn/mobile.html


In [2]:
url = "https://cellphones.com.vn/mobile.html"

def get_chrome_options():
    chrome_options = Options()

    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-extensions")
    chrome_options.add_argument("--disable-logging")
    chrome_options.add_argument("--disable-notifications")
    chrome_options.add_argument("--disable-web-security")
    prefs = {"profile.managed_default_content_settings.images": 2}
    chrome_options.add_experimental_option("prefs", prefs)
    chrome_options.page_load_strategy = "eager"
    return chrome_options

driver: WebDriver = webdriver.Chrome(
    service=Service(executable_path=ChromeDriverManager().install()),
    options=get_chrome_options(),
)
driver.get(url)


def remove_subscriber_popup(driver: WebDriver):
    try:
        button = driver.find_element(By.CLASS_NAME, "cancel-button-top")
        button.click()
    except Exception:
        pass

Hiển thị toàn bộ sản phẩm trước khi lấy link


In [3]:
button_class = "button__show-more-product"
def click_show_more_btn(button_class: str, max_attempts=20):
    attempts = 0
    while attempts < max_attempts:
        try:
            # Tìm button
            buttons = driver.find_elements(By.CLASS_NAME, button_class)
            if not buttons:
                print("No more 'Show More' buttons found")
                break

            # Lướt đến button
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", buttons[0])
            time.sleep(1)

            driver.execute_script("arguments[0].click();", buttons[0])

            print(f"Clicked button: {attempts+1} times")
            attempts += 1

            # Đợi nội dung load xong
            time.sleep(2)
        except Exception as e:
            print(f"Error: {e}")
            break

driver_wait = WebDriverWait(driver, 10)
button = driver_wait.until(
    EC.element_to_be_clickable((By.CLASS_NAME, button_class))
)
click_show_more_btn(button_class, max_attempts=25)

Clicked button: 1 times
Clicked button: 2 times
Clicked button: 3 times
Clicked button: 4 times
Clicked button: 5 times
Clicked button: 6 times
Clicked button: 7 times
Clicked button: 8 times
Clicked button: 9 times
Clicked button: 10 times
Clicked button: 11 times
Clicked button: 12 times
Clicked button: 13 times
Clicked button: 14 times
Clicked button: 15 times
Clicked button: 16 times
Clicked button: 17 times
Clicked button: 18 times
Clicked button: 19 times
Clicked button: 20 times
Clicked button: 21 times
Clicked button: 22 times
Clicked button: 23 times
Clicked button: 24 times
Clicked button: 25 times


In [4]:
remove_subscriber_popup(driver)
product_elements = driver.find_elements(By.CSS_SELECTOR, "div.product-info-container.product-item .product-info")
print(f"Tổng số sản phẩm: {len(product_elements)}")

Tổng số sản phẩm: 520


In [5]:
print(product_elements[0].text)

Samsung Galaxy S26 Ultra 12GB 256GB
Hàng mới về
32.990.000đ
36.990.000đ
Giảm 
11%
6.9 inches
12 GB
256 GB
Smember giảm đến 
330.000đ
S-Student giảm thêm 
500.000đ
Không phí chuyển đổi khi trả góp 0% qua thẻ tín dụng kỳ hạn 3-6 tháng


### Trích xuất các link sản phẩm từ trang web cellphones.com.vn/mobile.html dựa vào html của các thẻ sản phẩm


In [ ]:
product_links = []
link_lock = Lock()

def extract_product_link(product):
    try:
        # Tìm link sản phẩm
        link_element = product.find_element(By.CLASS_NAME, "product__link")
        link = link_element.get_attribute("href")
        
        with link_lock:
            product_links.append(link)
        
        return link
    except Exception as e:
        print(f"Error: {e} in product {product.text} - skipping")
        return None

num_threads = 12

with ThreadPoolExecutor(max_workers=num_threads) as executor:
    futures = [executor.submit(extract_product_link, product) for product in product_elements]
    
    wait(futures)

print(f"Found {len(product_links)} product links")
print(product_links[:5])

Found 520 product links
['https://cellphones.com.vn/dien-thoai-samsung-galaxy-s26-ultra.html', 'https://cellphones.com.vn/iphone-17-pro.html', 'https://cellphones.com.vn/iphone-17-pro-max.html', 'https://cellphones.com.vn/iphone-17-256gb.html', 'https://cellphones.com.vn/iphone-air-256gb.html']


### Sau khi lấy được các Link hợp lệ, ta tiến hành lưu vào file để đảm bảo không bị mất dữ liệu trong quá trình cào dữ liệu.


In [7]:
cellphones_links = "../link/product_links_cellphones.csv"

os.makedirs(os.path.dirname(cellphones_links), exist_ok=True)

with open(cellphones_links, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Product Link"])
    writer.writerows([[link] for link in product_links])

### Ta tiến hành cào dữ liệu thô, dữ liệu sẽ lưu theo cấu trúc JSON


In [ ]:
def extract_product_common_info(driver: WebDriver, product_link: str) -> dict[str, Any]:
    product_data = {
        "url": product_link,
        "name": "",
        "price": "",
        "specifications": []
    }

    print(f"Extracting data from: {product_link}")
    
    remove_subscriber_popup(driver)

    # Lấy tên sản phẩm
    try:
        name_element = driver.find_element(By.CSS_SELECTOR, ".box-product-name h1")
        name = name_element.text.strip()
        if name:
            product_data["name"] = name
            print(f"Found product name: {name}")
    except Exception as e:
        print(f"Error getting product name: {e}")

    time.sleep(1) 

    # Lấy giá sản phẩm
    try:
        price_element = driver.find_element(By.CSS_SELECTOR, "div.sale-price")
        price = price_element.text.strip()
        if price:
            product_data["price"] = price
            print(f"Found product price: {price}")
    except Exception as e:
        print(f"Error getting product price: {e}")

    # Trích xuất thông số kỹ thuật
    specifications = []

    try:
        driver.execute_script("document.querySelector('.button__show-modal-technical')?.click();")
        WebDriverWait(driver, 5).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "#teleport-modal .technical-content-section")
            )
        )
        time.sleep(1)

        sections = driver.find_elements(
            By.CSS_SELECTOR, ".technical-modal-container .technical-content-section"
        )

        for section in sections:
            # Lấy tên Category (VD: "Màn hình", "Camera sau")
            category_name = section.find_element(
                By.CSS_SELECTOR, "p.title"
            ).text.strip()

            details = {}

            # Lấy tất cả các hàng thông số trong table của section này
            rows = section.find_elements(
                By.CSS_SELECTOR, "tr.technical-content-item"
            )

            for row in rows:
                tds = row.find_elements(By.TAG_NAME, "td")
                if len(tds) >= 2:
                    key = tds[0].text.strip()
                    raw_value = tds[1].text.strip()

                    # XỬ LÝ ĐỂ TẠO ARRAY (LIST) CHO JSON
                    # Selenium .text tự động biến <br> hoặc các block element thành \n
                    if "\n" in raw_value:
                        # Tách chuỗi theo dòng, loại bỏ khoảng trắng dư thừa và bỏ qua dòng trống
                        value_list = [
                            item.strip()
                            for item in raw_value.split("\n")
                            if item.strip()
                        ]
                        details[key] = value_list
                    else:
                        details[key] = raw_value

            # Thêm vào mảng specifications nếu có dữ liệu
            if details:
                specifications.append(
                    {"category": category_name, "details": details}
                )

    except Exception as e:
        print(f"Error during specification extraction: {e}")

    driver.execute_script("document.querySelector('.close-btn')?.click();")

    if specifications:
        product_data["specifications"] = specifications

    print(f"product name common data: {product_data}")
    return product_data

### Lấy dữ liệu và lưu dữ liệu vào file json


In [50]:
json_product_detail_folder = "../data/json"
def getAndSaveProductInfo(driver: WebDriver, product_link: str) -> dict[str, Any]:
    """
    Lấy thông tin sản phẩm từ trang product_link bằng cách cào thông tin common và variant, sau đó merge lại thành 1 dict.
    
    Returns:
    - Dictionary chứa thông tin sản phẩm, bao gồm:
         - url, specifications, variant_prices
    """
    driver.get(product_link)
    time.sleep(2)
    
    product_common_data = extract_product_common_info(driver, product_link)
    if product_common_data is None:
        product_common_data = {"url": product_link, "specifications": {}}
    
    file_name = product_link.split("/")[-1].replace(".html", "")
    
    save_path = os.path.join(json_product_detail_folder, file_name + ".json")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(product_common_data, f, indent=4, ensure_ascii=False)
        
    return product_common_data

Test hàm với 1 link sản phẩm ngẫu nhiên triong file product_links.csv


In [44]:
getAndSaveProductInfo(driver, "https://cellphones.com.vn/iphone-16-pro-max.html")

Extracting data from: https://cellphones.com.vn/iphone-16-pro-max.html
Found product name: Điện thoại iPhone 16 Pro Max 256GB
Found product price: 31.390.000đ
product name common data: {'url': 'https://cellphones.com.vn/iphone-16-pro-max.html', 'name': 'Điện thoại iPhone 16 Pro Max 256GB', 'price': '31.390.000đ', 'specifications': [{'category': 'Màn hình', 'details': {'Kích thước màn hình': '6.9 inches', 'Công nghệ màn hình': 'Super Retina XDR OLED', 'Độ phân giải màn hình': '2868 x 1320 pixels', 'Tính năng màn hình': ['Dynamic Island', 'Màn hình Luôn Bật', 'Công nghệ ProMotion với tốc độ làm mới thích ứng lên đến 120Hz', 'Màn hình HDR', 'True Tone', 'Dải màu rộng (P3)', 'Haptic Touch', 'Tỷ lệ tương phản 2.000.000:1'], 'Tần số quét': '120Hz', 'Kiểu màn hình': 'Dynamic Island'}}, {'category': 'Camera sau', 'details': {'Camera sau': ['Camera chính: 48MP, f/1.78, 24mm, 2µm, chống rung quang học dịch chuyển cảm biến thế hệ thứ hai, Focus Pixels 100%', 'Telephoto 2x 12MP: 52 mm, ƒ/1.6', 'Ca

{'url': 'https://cellphones.com.vn/iphone-16-pro-max.html',
 'name': 'Điện thoại iPhone 16 Pro Max 256GB',
 'price': '31.390.000đ',
 'specifications': [{'category': 'Màn hình',
   'details': {'Kích thước màn hình': '6.9 inches',
    'Công nghệ màn hình': 'Super Retina XDR OLED',
    'Độ phân giải màn hình': '2868 x 1320 pixels',
    'Tính năng màn hình': ['Dynamic Island',
     'Màn hình Luôn Bật',
     'Công nghệ ProMotion với tốc độ làm mới thích ứng lên đến 120Hz',
     'Màn hình HDR',
     'True Tone',
     'Dải màu rộng (P3)',
     'Haptic Touch',
     'Tỷ lệ tương phản 2.000.000:1'],
    'Tần số quét': '120Hz',
    'Kiểu màn hình': 'Dynamic Island'}},
  {'category': 'Camera sau',
   'details': {'Camera sau': ['Camera chính: 48MP, f/1.78, 24mm, 2µm, chống rung quang học dịch chuyển cảm biến thế hệ thứ hai, Focus Pixels 100%',
     'Telephoto 2x 12MP: 52 mm, ƒ/1.6',
     'Camera góc siêu rộng: 48MP, 13 mm,ƒ/2.2 và trường ảnh 120°, Hybrid Focus Pixels, ảnh có độ phân giải'],
    'Qu

Lấy ra 5 link sản phẩm từ file

In [45]:
product_links = []
with open(cellphones_links, "r") as f:
    reader = csv.reader(f)
    next(reader)  # Skip header
    for row in reader:
        product_links.append(row[0])
product_links[:5]

['https://cellphones.com.vn/dien-thoai-samsung-galaxy-s26-ultra.html',
 'https://cellphones.com.vn/iphone-17-pro.html',
 'https://cellphones.com.vn/iphone-17-pro-max.html',
 'https://cellphones.com.vn/iphone-17-256gb.html',
 'https://cellphones.com.vn/iphone-air-256gb.html']

### Cuối cùng đây là hàm chính để chạy và cào dữ liệu, đảm bảo đã làm xong bước get Link và chạy các khai báo hàm rồi mới chạy được ở đây

-   Để giảm thời gian cào, ở đây nhóm sử dụng Đa luồng đa tuyến để cào dữ liệu. Lưu lại các Link lỗi để tiếp tục xử lí sau khi cào xong
-   Sử dụng 1 vài thư viện để log thông tin dễ theo dõi, debug trong quá trình cào dữ liệu


In [ ]:
def crawl_products_multithreaded(product_links: list[str], max_workers: int = 5, 
                                 batch_size: int = 10, delay_between_batches: int = 3) -> list[dict[str, Any]]:
    """
    Cào dữ liệu sản phẩm đa luồng từ danh sách các liên kết sản phẩm.
    
    Nếu một liên kết gặp lỗi, sẽ được thêm vào cuối danh sách để xử lý lại tuần tự sau.
    
    Args:
        product_links: Danh sách các URL sản phẩm cần cào
        max_workers: Số lượng luồng tối đa chạy đồng thời
        batch_size: Kích thước mỗi batch để tránh quá tải
        delay_between_batches: Thời gian chờ giữa các batch (giây)
    
    Returns:
        Danh sách các dictionary chứa thông tin sản phẩm
    """
    # Đảm bảo thư mục json tồn tại
    os.makedirs(json_product_detail_folder, exist_ok=True)
    
    results = []
    retry_links = []  # Danh sách các liên kết cần thử lại
    
    # Chia danh sách sản phẩm thành các batch nhỏ hơn
    batches = [product_links[i:i+batch_size] for i in range(0, len(product_links), batch_size)]
    
    print(f"Chia thành {len(batches)} batch, mỗi batch {batch_size} sản phẩm")
    
    def worker(link):
        try:
            # Khởi tạo driver riêng cho mỗi thread
            options = get_chrome_options()
            options.add_argument("--headless")
            
            driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), 
                                      options=options)
            
            try:
                # Random Sleep trước mỗi request để tránh bị chặn
                time.sleep(random.uniform(1, 3))
                # Gọi hàm getAndSaveProductInfo để cào dữ liệu
                result = getAndSaveProductInfo(driver, link)
                return result
            finally:
                # Đảm bảo đóng driver sau khi sử dụng
                driver.quit()
                
        except Exception as e:
            error_msg = f"Lỗi khi xử lý link {link}: {str(e)}"
            print(error_msg)
            return {"url": link, "error": str(e), "specifications": {}, "variant_prices": []}
    
    # Xử lý theo từng batch với đa luồng
    for batch_idx, batch in enumerate(batches):
        print(f"Đang xử lý batch {batch_idx+1}/{len(batches)}")
        batch_results = []
        
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(worker, link): link for link in batch}
            for future in tqdm(as_completed(futures), total=len(batch), 
                              desc=f"Batch {batch_idx+1}"):
                link = futures[future]
                try:
                    result = future.result()
                    batch_results.append(result)
                except Exception as e:
                    # Nếu có lỗi ngoài dự tính, báo lỗi và tiếp tục
                    print(f"Lỗi khi lấy kết quả cho {link}: {str(e)}")
                    batch_results.append({"url": link, "error": str(e), "specifications": {}, "variant_prices": []})
        
        results.extend(batch_results)
        print(f"Đã cào {len(batch_results)}/{len(batch)} sản phẩm từ batch {batch_idx+1}")
        print(f"Tổng số sản phẩm đã cào: {len(results)}/{len(product_links)}")
        
        if batch_idx < len(batches) - 1:
            print(f"Chờ {delay_between_batches} giây trước khi xử lý batch tiếp theo...")
            time.sleep(delay_between_batches)
    
    # Xử lý các liên kết lỗi sau khi đa luồng bằng cách tuần tự
    # Nếu một liên kết lỗi, thêm nó vào cuối danh sách retry_links để thử lại sau
    retry_links = [result["url"] for result in results if result.get("error")]
    attempt = 1
    while retry_links:
        print(f"\nBắt đầu xử lý lại {len(retry_links)} liên kết lỗi (Lần thử: {attempt})")
        current_retry = retry_links.copy()
        retry_links = []  # reset danh sách lỗi
        
        for link in tqdm(current_retry, desc=f"Retry attempt {attempt}"):
            try:
                options = get_chrome_options()
                options.add_argument("--headless")  # Chạy Chrome ngầm không hiển thị giao diện (tiết kiệm tài nguyên)
                
                driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), 
                                          options=options)
                try:
                    # Cho phép load chậm hơn, không thêm độ trễ ngẫu nhiên
                    time.sleep(2)
                    result = getAndSaveProductInfo(driver, link)
                    # Cập nhật kết quả mới vào danh sách results:
                    # Tìm vị trí của kết quả cũ và thay thế
                    for idx, r in enumerate(results):
                        if r["url"] == link:
                            results[idx] = result
                            break
                finally:
                    driver.quit()
            except Exception as e:
                print(f"Lỗi khi xử lý lại link {link}: {str(e)}")
                # Nếu lỗi vẫn xảy ra, thêm link vào cuối danh sách để thử lại sau
                retry_links.append(link)
        if retry_links:
            print(f"Vẫn còn {len(retry_links)} liên kết lỗi, chờ 10 giây trước khi thử lại...")
            time.sleep(10)
        attempt += 1

    # Lưu tất cả kết quả vào một file JSON
    try:
        with open("../data/raw/cellphones_data.json", "w", encoding="utf-8") as f:
            json.dump(results, f, indent=4, ensure_ascii=False)
        print(f"Đã lưu tất cả {len(results)} kết quả vào" + f"{"../data/raw/cellphones_data.json"}")
    except Exception as e:
        print(f"Lỗi khi lưu {"../data/raw/cellphones_data.json"}: {str(e)}")
    
    return results

results = crawl_products_multithreaded(
    product_links=product_links[:10],
    max_workers=5,
    batch_size=5,
    delay_between_batches=10
)

print(f"Hoàn thành cào dữ liệu cho {len(results)} sản phẩm")


Chia thành 2 batch, mỗi batch 5 sản phẩm
Đang xử lý batch 1/2


Batch 1:   0%|          | 0/5 [00:00<?, ?it/s]

Extracting data from: https://cellphones.com.vn/iphone-17-pro-max.html
Extracting data from: https://cellphones.com.vn/dien-thoai-samsung-galaxy-s26-ultra.html
Extracting data from: https://cellphones.com.vn/iphone-17-256gb.html
Extracting data from: https://cellphones.com.vn/iphone-air-256gb.html
Extracting data from: https://cellphones.com.vn/iphone-17-pro.html
Found product price: 24.990.000đ
Found product price: 32.990.000đ
Found product price: 37.990.000đ
Found product price: 24.690.000đ
Found product price: 34.590.000đ
product name common data: {'url': 'https://cellphones.com.vn/dien-thoai-samsung-galaxy-s26-ultra.html', 'name': '', 'price': '32.990.000đ', 'specifications': [{'category': 'Màn hình', 'details': {'Kích thước màn hình': '6.9 inches', 'Công nghệ màn hình': 'Dynamic AMOLED 2X', 'Độ phân giải màn hình': '3120 x 1440 pixels (Quad HD+)', 'Tính năng màn hình': ['Tần số quét: 1-120Hz', 'Độ sáng tối đa: 2600 nits'], 'Tần số quét': '120Hz', 'Kiểu màn hình': 'Đục lỗ (Nốt ruồi

Batch 1: 100%|██████████| 5/5 [02:40<00:00, 32.11s/it] 


Đã cào 5/5 sản phẩm từ batch 1
Tổng số sản phẩm đã cào: 5/10
Chờ 10 giây trước khi xử lý batch tiếp theo...
Đang xử lý batch 2/2


Batch 2:   0%|          | 0/5 [00:00<?, ?it/s]

Extracting data from: https://cellphones.com.vn/dien-thoai-samsung-galaxy-z-flip-7.html
Extracting data from: https://cellphones.com.vn/dien-thoai-samsung-galaxy-s26.html
Extracting data from: https://cellphones.com.vn/dien-thoai-oppo-reno15-f.html
Extracting data from: https://cellphones.com.vn/iphone-15.html
Extracting data from: https://cellphones.com.vn/dien-thoai-samsung-galaxy-z-fold-7.html
Found product price: 23.990.000đ
Found product price: 18.090.000đ
Found product price: 11.990.000đ
Found product price: 22.990.000đ
Found product price: 41.990.000đ
product name common data: {'url': 'https://cellphones.com.vn/dien-thoai-samsung-galaxy-z-flip-7.html', 'name': '', 'price': '23.990.000đ', 'specifications': [{'category': 'Màn hình', 'details': {'Kích thước màn hình': '6.9 inches', 'Công nghệ màn hình': 'Dynamic AMOLED 2X', 'Tính năng màn hình': ['Tần số quét: 120 Hz', 'Độ phân giải màn hình chính: 2520 x 1080 (FHD+)', 'Kích cỡ màn hình phụ: 4.1"', 'Độ phân giải màn hình phụ: 1048 

Batch 2: 100%|██████████| 5/5 [01:39<00:00, 19.81s/it]

Đã cào 5/5 sản phẩm từ batch 2
Tổng số sản phẩm đã cào: 10/10
Lỗi khi lưu ../data/processed/cellphones_data.json: [Errno 2] No such file or directory: '../data/processed/cellphones_data.json'
Hoàn thành cào dữ liệu cho 10 sản phẩm
